# Simulate the gene expression in a population of cells

The code simulates gene expression based on a GRN (described by the interaction matrix) and expression of each gene is defined by parameters (each row in the parameter sheet) using the Gillespie algorithm.


In [4]:
import numpy as np
root = "/projects/b1042/GoyalLab/Keerthana/"

base_config = {
    'n_cells': 5000, #Number of cells before division (number of twins)
    'time_points': np.arange(0, 2500, 1), #This is the time used to run the initial cells till they reach simulation (in hours)
    'sample_twins_time_points': np.arange(0, 49, 1), #This is the time twin cells are simulated after division and the 1 describes sampling frequency (all in hours)
    "path_to_matrix": f"{root}/grnInference/simulation_data/median_parameter_simulations/simulation_details/interaction_matrix_no_reg.txt", #path to the interaction matrix specifying the GRN to simulate
    "param_csv": f"{root}/grnInference/simulation_data/median_parameter_simulations/simulation_details/median_param.csv", #path to the parameters for all genes and interaction terms
    "rows_to_use": [[0, 1]], #Rows in the parameter csv to use for each gene - length should be equal to number of genes in the system
    "output_folder": f"{root}/grnInference/simulation_data/median_parameter_simulations/new_simulation/", #path to folder to store simulation 
    "log_file": f"{root}/grnInference/simulation_data/median_parameter_simulations/simulation_details/median_parameter_simulations.jsonl", #path to the log file
    "type": "A_B",  # name of the network used for interaction - will be in the filename
    "number_of_parallel_parameters": 2, #number of parameters to be run in parallel
    "number_of_cores_per_parameter": 12 #Number of cores to be used per parameter (number_of_parallel_parameters*number_of_cores_per_parameter = number of cores in your computer)
}



## Import functions from gillespie_script


In [3]:
from joblib import Parallel, delayed
from tqdm import tqdm
import os
from tqdm import tqdm
import sys


def add_gillespie_script_to_path(target_dir_name="grnInferenceRepo"):
    current = os.path.abspath(__file__) if "__file__" in globals() else os.getcwd()
    while True:
        if os.path.basename(current) == target_dir_name:
            target_path = os.path.join(current, "simulation_scripts", "parameter_scan")
            if target_path not in sys.path:
                sys.path.append(target_path)
            return
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError(f"Could not find repo directory '{target_dir_name}'")
        current = parent

# ✅ Call it
add_gillespie_script_to_path()

# ✅ Now import works
from gillespie_script import process_param_set


from gillespie_script import process_param_set
from numba import set_num_threads, get_num_threads

set_num_threads(base_config['number_of_cores_per_parameter'])
print("Threads Numba will use:", get_num_threads())

Threads Numba will use: 12


# Start running the simulation


In [8]:
os.makedirs(base_config['output_folder'], exist_ok=True)
rows_to_use = base_config['rows_to_use']
labels = ["rows_" + "_".join(map(str, row)) for row in rows_to_use]
param_sets = list(zip(rows_to_use, labels))

# ✅ Run over specified number of jobs (workers)
results = Parallel(n_jobs=base_config['number_of_parallel_parameters'])(
    delayed(process_param_set)(rows, label, base_config)
    for rows, label in tqdm(param_sets, total=len(param_sets), desc="All iterations")
)

# ✅ Print results (matching the concurrent.futures style)
for (rows, label), res in zip(param_sets, results):
    print(f"Completed simulation for {label} (rows={rows}): {res}")

All iterations:   0%|          | 0/1 [00:00<?, ?it/s]

[Worker rows_0_1] Using 12 threads for rows=[0, 1]

{'{p_on_gene_1}': 0.66, '{p_off_gene_1}': 8.8, '{p_prod_mRNA_gene_1}': 2.0, '{p_prod_protein_gene_1}': 560.0, '{n_gene_1_to_gene_2}': 2.0, '{p_add_gene_1_to_gene_2}': 6.0, '{n_gene_2_to_gene_1}': 2.0, '{p_add_gene_2_to_gene_1}': 6.0, '{n_gene_1_to_gene_3}': 2.0, '{p_add_gene_1_to_gene_3}': 6.0, '{n_gene_2_to_gene_3}': 2.0, '{p_add_gene_2_to_gene_3}': 6.0, '{pair_id_gene_1}': 0.0, '{gene_id_gene_1}': 1.0, '{p_deg_mRNA_gene_1}': 0.17328679513998632, '{p_deg_protein_gene_1}': 0.015403270679109895, '{p_on_gene_2}': 0.66, '{p_off_gene_2}': 8.8, '{p_prod_mRNA_gene_2}': 2.0, '{p_prod_protein_gene_2}': 560.0, '{pair_id_gene_2}': 0.0, '{gene_id_gene_2}': 2.0, '{p_deg_mRNA_gene_2}': 0.17328679513998632, '{p_deg_protein_gene_2}': 0.015403270679109895}
